In [47]:
import torch

batch_size = 2
channels = 4
num_heads = 2 # 2
head_size = channels // num_heads # 2

token_length = 3 # not part of hyperparameter

# 2 * 2 * 3 * 2 * 3 = 72
qkv = torch.arange(batch_size * token_length * num_heads * head_size * 3).view(batch_size, token_length, channels * 3)
q, k, v = qkv.split(channels, dim=-1)
print(f"qkv:\n{qkv}")
print(f"q before view and transpose:\n{q}")
q = q.view(batch_size, token_length, num_heads, head_size).transpose(1,2)
k = k.view(batch_size, token_length, num_heads, head_size).transpose(1,2)
v = v.view(batch_size, token_length, num_heads, head_size).transpose(1,2)
print(f"q:\n{q}")
print(f"q[:][0]:\n{q[:][0]}")
print(f"k:\n{k}")
print(f"v:\n{v}")

qkv:
tensor([[[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
         [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23],
         [24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35]],

        [[36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47],
         [48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59],
         [60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71]]])
q before view and transpose:
tensor([[[ 0,  1,  2,  3],
         [12, 13, 14, 15],
         [24, 25, 26, 27]],

        [[36, 37, 38, 39],
         [48, 49, 50, 51],
         [60, 61, 62, 63]]])
q:
tensor([[[[ 0,  1],
          [12, 13],
          [24, 25]],

         [[ 2,  3],
          [14, 15],
          [26, 27]]],


        [[[36, 37],
          [48, 49],
          [60, 61]],

         [[38, 39],
          [50, 51],
          [62, 63]]]])
q[:][0]:
tensor([[[ 0,  1],
         [12, 13],
         [24, 25]],

        [[ 2,  3],
         [14, 15],
         [26, 27]]])
k:
tensor([[[[ 4,  5],
          [16, 17],
          [28, 29]

In [ ]:
print(f"k.size(): {k.size()}") # batch_size, num_heads, token_length, head_size
print(f"k.transpose(-2, -1).size(): {k.transpose(-2, -1).size()}") # batch_size, num_heads, head_size, token_length

k.size(): torch.Size([2, 2, 3, 2])
k.transpose(-2, -1).size(): torch.Size([2, 2, 2, 3])


In [ ]:
import math

att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(head_size)) # batch_size, num_heads, token_length, token_length
att

tensor([[[[3.5355e+00, 1.2021e+01, 2.0506e+01],
          [7.9903e+01, 2.9204e+02, 5.0417e+02],
          [1.5627e+02, 5.7205e+02, 9.8783e+02]],

         [[2.3335e+01, 6.5761e+01, 1.0819e+02],
          [1.3364e+02, 3.7972e+02, 6.2579e+02],
          [2.4395e+02, 6.9367e+02, 1.1434e+03]]],


        [[[2.0909e+03, 2.7103e+03, 3.3298e+03],
          [2.7782e+03, 3.6013e+03, 4.4244e+03],
          [3.4655e+03, 4.4922e+03, 5.5190e+03]],

         [[2.3144e+03, 2.9677e+03, 3.6211e+03],
          [3.0356e+03, 3.8926e+03, 4.7496e+03],
          [3.7569e+03, 4.8175e+03, 5.8782e+03]]]])

In [45]:
block_size = 4 # max length of sequence

bias = torch.tril(torch.ones(block_size, block_size)).view(1, 1, block_size, block_size)
print(f"bias:\n{bias}")
print(f"bias[:,:,:T,:T]:\n{bias[:,:,:token_length,:token_length]}")

bias:
tensor([[[[1., 0., 0., 0.],
          [1., 1., 0., 0.],
          [1., 1., 1., 0.],
          [1., 1., 1., 1.]]]])
bias[:,:,:T,:T]:
tensor([[[[1., 0., 0.],
          [1., 1., 0.],
          [1., 1., 1.]]]])


In [50]:
model_type = 'gpt2'

config_args = {
  'gpt2': {'n_layer': 12, 'n_head': 12, 'n_embd': 768},         # 124M params
  'gpt2-medium': {'n_layer': 24, 'n_head': 16, 'n_embd': 1024}, # 350M params
  'gpt2-large': {'n_layer': 36, 'n_head': 20, 'n_embd': 1280},  # 774M params
  'gpt2-xl': {'n_layer': 48, 'n_head': 25, 'n_embd': 1600}      # 1558M params
}[model_type]
config_args['vocab_size'] = 50257 # always 50257 for GPT model checkpoints
config_args['block_size'] = 1024 # always 1024 for GPT model checkpoints
config_args

{'n_layer': 12,
 'n_head': 12,
 'n_embd': 768,
 'vocab_size': 50257,
 'block_size': 1024}